# 10_03.Association Rule
- pip install mlxtend
- http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/


## 1.기본 package 설정

In [1]:
## 기본
import numpy as np  # numpy 패키지 가져오기
import pandas as pd # pandas 패키지 가져오기

## Unsupervised 모델 (책에 없음)
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

## 2.데이터 불러오기

### 2.1 구글 드라이브와 연결

In [2]:
from google.colab import drive
drive.mount('/content/drive')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/pexpect/popen_spawn.py:60: DeprecationWarning: setDaemon() is deprecated, set the daemon attribute instead
  self._read_thread.setDaemon(True)


Mounted at /content/drive


### 2.2 데이터 프레임으로 저장
- 원본데이터(csv)를 dataframe 형태로 가져오기(pandas)

In [3]:
books_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CharlesBookClub.csv')
books_df.head()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Seq#,ID#,Gender,M,R,F,FirstPurch,ChildBks,YouthBks,CookBks,...,ItalCook,ItalAtlas,ItalArt,Florence,Related Purchase,Mcode,Rcode,Fcode,Yes_Florence,No_Florence
0,1,25,1,297,14,2,22,0,1,1,...,0,0,0,0,0,5,4,2,0,1
1,2,29,0,128,8,2,10,0,0,0,...,0,0,0,0,0,4,3,2,0,1
2,3,46,1,138,22,7,56,2,1,2,...,1,0,0,0,2,4,4,3,0,1
3,4,47,1,228,2,1,2,0,0,0,...,0,0,0,0,0,5,1,1,0,1
4,5,51,1,257,10,1,10,0,0,0,...,0,0,0,0,0,5,3,1,0,1


### 2.3 자료구조 살펴보기

In [ ]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Seq#              4000 non-null   int64
 1   ID#               4000 non-null   int64
 2   Gender            4000 non-null   int64
 3   M                 4000 non-null   int64
 4   R                 4000 non-null   int64
 5   F                 4000 non-null   int64
 6   FirstPurch        4000 non-null   int64
 7   ChildBks          4000 non-null   int64
 8   YouthBks          4000 non-null   int64
 9   CookBks           4000 non-null   int64
 10  DoItYBks          4000 non-null   int64
 11  RefBks            4000 non-null   int64
 12  ArtBks            4000 non-null   int64
 13  GeogBks           4000 non-null   int64
 14  ItalCook          4000 non-null   int64
 15  ItalAtlas         4000 non-null   int64
 16  ItalArt           4000 non-null   int64
 17  Florence          4000 non-null  

In [ ]:
books_df.columns

Index(['Seq#', 'ID#', 'Gender', 'M', 'R', 'F', 'FirstPurch', 'ChildBks',
       'YouthBks', 'CookBks', 'DoItYBks', 'RefBks', 'ArtBks', 'GeogBks',
       'ItalCook', 'ItalAtlas', 'ItalArt', 'Florence', 'Related Purchase',
       'Mcode', 'Rcode', 'Fcode', 'Yes_Florence', 'No_Florence'],
      dtype='object')

## 3.데이터 탐색

### 3.1 필요없는 변수 제거

In [4]:
books_df = books_df.iloc[:, 7:18]
books_df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence
0,0,1,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,2,1,2,0,1,0,1,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
3995,0,0,0,0,0,0,0,0,0,0,0
3996,1,1,2,2,2,0,1,0,0,0,0
3997,0,0,0,0,0,0,0,0,0,0,0
3998,1,1,3,1,0,0,0,0,0,0,0


### 3.2 구매 1 이상인 것은 1로 수정

In [5]:
books_df = (books_df > 0).astype(int)
books_df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence
0,0,1,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,0,1,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
3995,0,0,0,0,0,0,0,0,0,0,0
3996,1,1,1,1,1,0,1,0,0,0,0
3997,0,0,0,0,0,0,0,0,0,0,0
3998,1,1,1,1,0,0,0,0,0,0,0


## 4.연관규칙

### 4.1 transactions으로 변환

In [8]:
freq_itemsets = apriori(books_df,
                        min_support = 0.01,
                        use_colnames = True)
freq_itemsets

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/mlxtend/frequent_patterns/fpcommon.py:110: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.39400,(ChildBks)
1,0.23825,(YouthBks)
2,0.41550,(CookBks)
3,0.25475,(DoItYBks)
4,0.20475,(RefBks)
...,...,...
283,0.01325,"(ChildBks, RefBks, YouthBks, DoItYBks, CookBks..."
284,0.01725,"(ChildBks, RefBks, YouthBks, DoItYBks, CookBks..."
285,0.01350,"(ChildBks, YouthBks, DoItYBks, CookBks, GeogBk..."
286,0.01300,"(ChildBks, YouthBks, RefBks, GeogBks, CookBks,..."


### 4.2 연관규칙 실행

In [9]:
rules = association_rules(freq_itemsets,
                          metric="lift",
                          min_threshold=1)
rules

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(YouthBks),(ChildBks),0.23825,0.39400,0.14750,0.619098,1.571314,0.053629,1.590959,0.477309
1,(ChildBks),(YouthBks),0.39400,0.23825,0.14750,0.374365,1.571314,0.053629,1.217564,0.599983
2,(ChildBks),(CookBks),0.39400,0.41550,0.24200,0.614213,1.478251,0.078293,1.515086,0.533869
3,(CookBks),(ChildBks),0.41550,0.39400,0.24200,0.582431,1.478251,0.078293,1.451256,0.553507
4,(ChildBks),(DoItYBks),0.39400,0.25475,0.16150,0.409898,1.609022,0.061129,1.262918,0.624595
...,...,...,...,...,...,...,...,...,...,...
3135,(RefBks),"(ChildBks, GeogBks, DoItYBks, CookBks, ArtBks)",0.20475,0.02275,0.01225,0.059829,2.629849,0.007592,1.039439,0.779315
3136,(DoItYBks),"(ChildBks, RefBks, GeogBks, CookBks, ArtBks)",0.25475,0.01975,0.01225,0.048086,2.434752,0.007219,1.029768,0.790715
3137,(CookBks),"(ChildBks, RefBks, DoItYBks, GeogBks, ArtBks)",0.41550,0.01475,0.01225,0.029483,1.998817,0.006121,1.015180,0.854926
3138,(GeogBks),"(ChildBks, RefBks, DoItYBks, CookBks, ArtBks)",0.26675,0.02275,0.01225,0.045923,2.018600,0.006181,1.024289,0.688179


## 5.연관규칙 확인


### 5.1 lift가 높은 순서로 sorting

In [10]:
rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules = rules.sort_values('lift', ascending=[False])
rules

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,support,confidence,lift
1876,"(ItalCook, ArtBks)","(ItalArt, CookBks)",0.01150,0.335766,10.575320
1881,"(ItalArt, CookBks)","(ItalCook, ArtBks)",0.01150,0.362205,10.575320
1884,(ItalArt),"(ItalCook, CookBks, ArtBks)",0.01150,0.265896,10.530533
1873,"(ItalCook, CookBks, ArtBks)",(ItalArt),0.01150,0.455446,10.530533
737,(ItalArt),"(ItalCook, ArtBks)",0.01250,0.289017,8.438463
...,...,...,...,...,...
64,(Florence),(DoItYBks),0.02375,0.281065,1.103298
561,(Florence),"(RefBks, CookBks)",0.01275,0.150888,1.079696
560,"(RefBks, CookBks)",(Florence),0.01275,0.091234,1.079696
525,(Florence),"(DoItYBks, CookBks)",0.01500,0.177515,1.051940


### 5.2 intem 갯수 확인하기

In [11]:
rules["antecedent_len"] = rules["antecedents"].apply(lambda x: len(x))
rules

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,support,confidence,lift,antecedent_len
1876,"(ItalCook, ArtBks)","(ItalArt, CookBks)",0.01150,0.335766,10.575320,2
1881,"(ItalArt, CookBks)","(ItalCook, ArtBks)",0.01150,0.362205,10.575320,2
1884,(ItalArt),"(ItalCook, CookBks, ArtBks)",0.01150,0.265896,10.530533,1
1873,"(ItalCook, CookBks, ArtBks)",(ItalArt),0.01150,0.455446,10.530533,3
737,(ItalArt),"(ItalCook, ArtBks)",0.01250,0.289017,8.438463,1
...,...,...,...,...,...,...
64,(Florence),(DoItYBks),0.02375,0.281065,1.103298,1
561,(Florence),"(RefBks, CookBks)",0.01275,0.150888,1.079696,1
560,"(RefBks, CookBks)",(Florence),0.01275,0.091234,1.079696,2
525,(Florence),"(DoItYBks, CookBks)",0.01500,0.177515,1.051940,1


### 5.3 Multi 규칙확인

In [15]:
rules[ (rules['antecedent_len'] >= 2) &
       (rules['support'] > 0.1) &
       (rules['confidence'] > 0.7) &
       (rules['lift'] > 1) ]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,support,confidence,lift,antecedent_len
102,"(YouthBks, ChildBks)",(CookBks),0.12000,0.813559,1.958025,2
162,"(RefBks, ChildBks)",(CookBks),0.10350,0.807018,1.942280,2
158,"(DoItYBks, CookBks)",(ChildBks),0.12775,0.757037,1.921414,2
156,"(ChildBks, DoItYBks)",(CookBks),0.12775,0.791022,1.903783,2
103,"(YouthBks, CookBks)",(ChildBks),0.12000,0.745342,1.891730,2
163,"(RefBks, CookBks)",(ChildBks),0.10350,0.740608,1.879716,2
174,"(ChildBks, GeogBks)",(CookBks),0.10950,0.748718,1.801969,2
176,"(GeogBks, CookBks)",(ChildBks),0.10950,0.700800,1.778680,2


### 5.4 특정 규칙 확인

In [17]:
rules[rules["antecedents"] == {"CookBks", "DoItYBks"}]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,support,confidence,lift,antecedent_len
2883,"(DoItYBks, CookBks)","(ArtBks, RefBks, ChildBks, YouthBks)",0.01325,0.078519,3.489712,2
2042,"(DoItYBks, CookBks)","(ItalArt, YouthBks, ChildBks)",0.01000,0.059259,3.435319,2
2462,"(DoItYBks, CookBks)","(ItalAtlas, RefBks, ChildBks)",0.01000,0.059259,3.386243,2
2704,"(DoItYBks, CookBks)","(RefBks, GeogBks, YouthBks)",0.02150,0.127407,3.287933,2
2945,"(DoItYBks, CookBks)","(RefBks, ChildBks, GeogBks, YouthBks)",0.01725,0.102222,3.245150,2
1083,"(DoItYBks, CookBks)","(ItalAtlas, ChildBks)",0.01350,0.080000,3.232323,2
1797,"(DoItYBks, CookBks)","(ItalArt, GeogBks)",0.01075,0.063704,3.225504,2
2372,"(DoItYBks, CookBks)","(ArtBks, RefBks, ChildBks)",0.02275,0.134815,3.172113,2
1924,"(DoItYBks, CookBks)","(RefBks, ChildBks, YouthBks)",0.03300,0.195556,3.154122,2
1097,"(DoItYBks, CookBks)","(ItalArt, ChildBks)",0.01675,0.099259,3.126276,2
